# 32. Price Anomaly Model Pickle 생성

## 목적
검증된 가격 이상치 탐지 모델을 프로덕션용 Pickle 파일로 패키징합니다.

## 입력 파일
- `data/processed/price/price_anomaly_detector.pkl` (31번 노트북 출력)
- `data/processed/price/anomaly_detection_results.pkl` (31번 노트북 출력)

## 출력 파일
- `pred/models/price_anomaly_v1.pkl`

## Pickle 구조
```python
{
    'version': '1.0.0',
    'created_at': datetime,
    'model_type': 'price_anomaly_detection',
    'metadata': {...},
    'components': {
        'category_statistics': dict,
        'global_statistics': dict,
        'thresholds': dict
    },
    'hyperparameters': {...}
}
```

In [1]:
# 필수 라이브러리 임포트
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("라이브러리 로드 완료")
print(f"현재 시간: {datetime.now()}")

라이브러리 로드 완료
현재 시간: 2025-12-15 01:44:15.133951


## 1. 입력 파일 로드

In [2]:
# 경로 설정
INPUT_DIR = Path('../data/processed/price')
OUTPUT_DIR = Path('../pred/models')

DETECTOR_PATH = INPUT_DIR / 'price_anomaly_detector.pkl'
RESULTS_PATH = INPUT_DIR / 'anomaly_detection_results.pkl'

# 출력 디렉토리 생성
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"입력 디렉토리: {INPUT_DIR}")
print(f"출력 디렉토리: {OUTPUT_DIR}")

입력 디렉토리: ..\data\processed\price
출력 디렉토리: ..\pred\models


In [3]:
# 탐지기 상태 로드
if DETECTOR_PATH.exists():
    with open(DETECTOR_PATH, 'rb') as f:
        detector_state = pickle.load(f)
    print(f"탐지기 상태 로드 완료: {DETECTOR_PATH}")
    print(f"  버전: {detector_state.get('version', 'unknown')}")
    print(f"  카테고리 수: {len(detector_state.get('category_stats', {}))}")
    LOAD_SUCCESS = True
else:
    print(f"[경고] 탐지기 상태 파일이 없습니다: {DETECTOR_PATH}")
    print("31번 노트북을 먼저 실행하거나 시뮬레이션 데이터를 생성합니다.")
    
    # 시뮬레이션 탐지기 상태 생성
    detector_state = {
        'version': '1.0.0',
        'created_at': datetime.now(),
        'category_stats': {
            '신선식품': {'log_median': 3.9, 'log_mad': 0.15, 'price_median': 8000, 'price_mean': 8500, 'n_products': 100},
            '유제품': {'log_median': 3.7, 'log_mad': 0.12, 'price_median': 5000, 'price_mean': 5200, 'n_products': 80},
            '음료': {'log_median': 3.5, 'log_mad': 0.18, 'price_median': 3000, 'price_mean': 3500, 'n_products': 120},
            '과자/스낵': {'log_median': 3.6, 'log_mad': 0.2, 'price_median': 4000, 'price_mean': 4200, 'n_products': 90},
            '냉동식품': {'log_median': 3.85, 'log_mad': 0.16, 'price_median': 7000, 'price_mean': 7500, 'n_products': 70}
        },
        'warning_threshold': 2.5,
        'danger_threshold': 3.5,
        'use_log': True,
        'global_stats': {'log_median': 3.7, 'log_mad': 0.17}
    }
    LOAD_SUCCESS = False
    print("시뮬레이션 탐지기 상태 생성 완료")

탐지기 상태 로드 완료: ..\data\processed\price\price_anomaly_detector.pkl
  버전: 1.0.0
  카테고리 수: 13


In [4]:
# 탐지 결과 로드 (선택적)
if RESULTS_PATH.exists():
    detection_results = pd.read_pickle(RESULTS_PATH)
    print(f"탐지 결과 로드 완료: {RESULTS_PATH}")
    print(f"  총 제품 수: {len(detection_results)}")
    
    # 요약 통계
    n_anomalies = detection_results['is_anomaly'].sum()
    print(f"  이상치 수: {n_anomalies} ({n_anomalies/len(detection_results)*100:.1f}%)")
else:
    print(f"[참고] 탐지 결과 파일이 없습니다 (선택 사항)")
    detection_results = None

탐지 결과 로드 완료: ..\data\processed\price\anomaly_detection_results.pkl
  총 제품 수: 1600
  이상치 수: 72 (4.5%)


## 2. 데이터 검증

In [5]:
# 필수 컴포넌트 검증
REQUIRED_KEYS = ['category_stats', 'warning_threshold', 'danger_threshold', 'use_log']
REQUIRED_STAT_KEYS = ['log_median', 'log_mad']

print("필수 컴포넌트 검증:")

# 최상위 키 검증
for key in REQUIRED_KEYS:
    if key in detector_state:
        print(f"  ✅ {key}")
    else:
        print(f"  ❌ {key} - 누락됨")
        raise KeyError(f"필수 키 누락: {key}")

# 카테고리 통계 검증
category_stats = detector_state['category_stats']
print(f"\n카테고리 통계 검증 ({len(category_stats)}개 카테고리):")

for cat_name, stats in list(category_stats.items())[:3]:  # 처음 3개만 표시
    missing = [k for k in REQUIRED_STAT_KEYS if k not in stats]
    if missing:
        print(f"  ❌ {cat_name}: 누락된 키 - {missing}")
    else:
        print(f"  ✅ {cat_name}")

print(f"  ... ({len(category_stats) - 3}개 더)")

print("\n모든 필수 컴포넌트 검증 완료")

필수 컴포넌트 검증:
  ✅ category_stats
  ✅ warning_threshold
  ✅ danger_threshold
  ✅ use_log

카테고리 통계 검증 (13개 카테고리):
  ✅ 견과/건과/간식
  ✅ 과일
  ✅ 김치/반찬/절임
  ... (10개 더)

모든 필수 컴포넌트 검증 완료


## 3. Pickle 구조 생성

In [6]:
# Pickle 구조 생성
MODEL_VERSION = '1.0.0'

# 하이퍼파라미터
hyperparameters = {
    'warning_threshold': detector_state['warning_threshold'],
    'danger_threshold': detector_state['danger_threshold'],
    'use_log_transform': detector_state['use_log'],
    'mad_scaling_factor': 0.6745,  # 정규분포 가정 상수
    'method': 'modified_z_score'
}

# 메타데이터
metadata = {
    'n_categories': len(category_stats),
    'categories': list(category_stats.keys()),
    'training_date': detector_state.get('created_at', datetime.now()).strftime('%Y-%m-%d'),
    'source_data': 'SelF products_product'
}

# 탐지 결과 요약 (있는 경우)
if detection_results is not None:
    n_total = len(detection_results)
    metadata['evaluation'] = {
        'n_products_evaluated': n_total,
        'n_normal': int((detection_results['anomaly_level'] == 0).sum()),
        'n_warning': int((detection_results['anomaly_level'] == 1).sum()),
        'n_danger': int((detection_results['anomaly_level'] == 2).sum()),
        'anomaly_rate': float(detection_results['is_anomaly'].mean())
    }

# 최종 Pickle 구조
price_anomaly_model = {
    'version': MODEL_VERSION,
    'created_at': datetime.now(),
    'model_type': 'price_anomaly_detection',
    'description': '가격 이상치 탐지 모델 (Modified Z-Score + 카테고리별 통계)',
    
    'metadata': metadata,
    
    'components': {
        # 카테고리별 통계
        'category_statistics': category_stats,
        
        # 글로벌 폴백 통계
        'global_statistics': detector_state.get('global_stats', {}),
        
        # 임계값
        'thresholds': {
            'warning': hyperparameters['warning_threshold'],
            'danger': hyperparameters['danger_threshold']
        }
    },
    
    'hyperparameters': hyperparameters
}

print("Pickle 구조 생성 완료")
print(f"\n모델 정보:")
print(f"  버전: {MODEL_VERSION}")
print(f"  카테고리 수: {metadata['n_categories']}")
print(f"  경고 임계값: {hyperparameters['warning_threshold']}")
print(f"  위험 임계값: {hyperparameters['danger_threshold']}")

Pickle 구조 생성 완료

모델 정보:
  버전: 1.0.0
  카테고리 수: 13
  경고 임계값: 2.5
  위험 임계값: 3.5


## 4. Pickle 저장

In [7]:
# Pickle 파일 저장
OUTPUT_PATH = OUTPUT_DIR / 'price_anomaly_v1.pkl'

with open(OUTPUT_PATH, 'wb') as f:
    pickle.dump(price_anomaly_model, f, protocol=pickle.HIGHEST_PROTOCOL)

# 파일 크기 확인
file_size_kb = OUTPUT_PATH.stat().st_size / 1024

print(f"Pickle 저장 완료: {OUTPUT_PATH}")
print(f"파일 크기: {file_size_kb:.2f} KB")

Pickle 저장 완료: ..\pred\models\price_anomaly_v1.pkl
파일 크기: 4.60 KB


## 5. 저장된 Pickle 검증

In [8]:
# 저장된 파일 다시 로드하여 검증
print("저장된 Pickle 파일 검증 중...")

with open(OUTPUT_PATH, 'rb') as f:
    loaded_model = pickle.load(f)

# 구조 검증
REQUIRED_TOP_KEYS = ['version', 'created_at', 'metadata', 'components', 'hyperparameters']
REQUIRED_COMPONENT_KEYS = ['category_statistics', 'global_statistics', 'thresholds']

print("\n구조 검증:")
for key in REQUIRED_TOP_KEYS:
    if key in loaded_model:
        print(f"  ✅ {key}")
    else:
        print(f"  ❌ {key}")

print("\n컴포넌트 검증:")
for key in REQUIRED_COMPONENT_KEYS:
    if key in loaded_model['components']:
        print(f"  ✅ components.{key}")
    else:
        print(f"  ❌ components.{key}")

저장된 Pickle 파일 검증 중...

구조 검증:
  ✅ version
  ✅ created_at
  ✅ metadata
  ✅ components
  ✅ hyperparameters

컴포넌트 검증:
  ✅ components.category_statistics
  ✅ components.global_statistics
  ✅ components.thresholds


In [9]:
# 이상치 탐지 함수 테스트
def detect_price_anomaly(model, price, category_name=None):
    """
    가격 이상치 탐지
    
    Args:
        model: 로드된 Pickle 모델
        price: 가격
        category_name: 카테고리명
    
    Returns:
        dict: 탐지 결과
    """
    components = model['components']
    hyperparams = model['hyperparameters']
    
    # 유효성 검사
    if price is None or price <= 0:
        return {
            'is_anomaly': True,
            'level': 2,  # DANGER
            'modified_zscore': float('inf'),
            'message': '유효하지 않은 가격'
        }
    
    # 통계 선택
    if category_name and category_name in components['category_statistics']:
        stats = components['category_statistics'][category_name]
    elif components['global_statistics']:
        stats = components['global_statistics']
    else:
        return {
            'is_anomaly': False,
            'level': 0,
            'modified_zscore': 0,
            'message': '통계 정보 없음'
        }
    
    # 로그 변환
    if hyperparams['use_log_transform']:
        log_price = np.log10(max(1, price))
    else:
        log_price = price
    
    # Modified Z-Score 계산
    median = stats['log_median']
    mad = stats['log_mad']
    
    if mad == 0:
        mad = 0.0001
    
    scaling = hyperparams['mad_scaling_factor']
    modified_zscore = scaling * (log_price - median) / mad
    abs_zscore = abs(modified_zscore)
    
    # 레벨 판정
    warning_thresh = components['thresholds']['warning']
    danger_thresh = components['thresholds']['danger']
    
    if abs_zscore >= danger_thresh:
        level = 2  # DANGER
        is_anomaly = True
    elif abs_zscore >= warning_thresh:
        level = 1  # WARNING
        is_anomaly = True
    else:
        level = 0  # NORMAL
        is_anomaly = False
    
    # 방향
    if modified_zscore > warning_thresh:
        direction = 'high'
    elif modified_zscore < -warning_thresh:
        direction = 'low'
    else:
        direction = 'normal'
    
    return {
        'is_anomaly': is_anomaly,
        'level': level,
        'modified_zscore': modified_zscore,
        'direction': direction
    }


# 테스트 실행
print("이상치 탐지 기능 테스트:")

test_cases = [
    ('신선식품', 8000),
    ('신선식품', 80000),
    ('신선식품', 500),
    ('음료', 3000),
    ('음료', 50000),
    (None, 5000)
]

level_names = ['정상', '경고', '위험']

for category, price in test_cases:
    result = detect_price_anomaly(loaded_model, price, category)
    level_str = level_names[result['level']]
    print(f"\n카테고리: {category}, 가격: {price:,}원")
    print(f"  레벨: {level_str}, Z-Score: {result['modified_zscore']:.2f}")

이상치 탐지 기능 테스트:

카테고리: 신선식품, 가격: 8,000원
  레벨: 정상, Z-Score: 0.48

카테고리: 신선식품, 가격: 80,000원
  레벨: 위험, Z-Score: 4.31

카테고리: 신선식품, 가격: 500원
  레벨: 위험, Z-Score: -4.13

카테고리: 음료, 가격: 3,000원
  레벨: 정상, Z-Score: -1.23

카테고리: 음료, 가격: 50,000원
  레벨: 경고, Z-Score: 2.85

카테고리: None, 가격: 5,000원
  레벨: 정상, Z-Score: -0.30


In [10]:
# 탐지 속도 벤치마크
import time

print("탐지 속도 벤치마크:")

n_tests = 1000
categories = list(loaded_model['components']['category_statistics'].keys())

# 랜덤 테스트 데이터 생성
np.random.seed(42)
test_prices = np.random.uniform(100, 100000, n_tests)
test_categories = np.random.choice(categories, n_tests)

start_time = time.time()
for price, category in zip(test_prices, test_categories):
    _ = detect_price_anomaly(loaded_model, price, category)
elapsed = time.time() - start_time

avg_time_ms = (elapsed / n_tests) * 1000
throughput = n_tests / elapsed

print(f"  테스트 횟수: {n_tests}")
print(f"  총 소요 시간: {elapsed:.3f}초")
print(f"  평균 탐지 시간: {avg_time_ms:.4f}ms")
print(f"  처리량: {throughput:,.0f} 건/초")

# 성능 기준 확인 (1ms 이하)
if avg_time_ms < 1:
    print(f"  ✅ 성능 기준 통과 (< 1ms)")
else:
    print(f"  ⚠️ 성능 개선 필요 (> 1ms)")

탐지 속도 벤치마크:
  테스트 횟수: 1000
  총 소요 시간: 0.001초
  평균 탐지 시간: 0.0010ms
  처리량: 995,090 건/초
  ✅ 성능 기준 통과 (< 1ms)


## 6. 모델 정보 요약

In [11]:
# 최종 요약
print("=" * 60)
print("Price Anomaly Model Pickle 생성 완료")
print("=" * 60)

print(f"\n📁 출력 파일: {OUTPUT_PATH}")
print(f"📊 파일 크기: {file_size_kb:.2f} KB")

print(f"\n📈 모델 정보:")
print(f"   버전: {loaded_model['version']}")
print(f"   생성일: {loaded_model['created_at']}")
print(f"   카테고리 수: {loaded_model['metadata']['n_categories']}")

print(f"\n⚙️ 하이퍼파라미터:")
for key, value in loaded_model['hyperparameters'].items():
    print(f"   {key}: {value}")

if 'evaluation' in loaded_model['metadata']:
    eval_info = loaded_model['metadata']['evaluation']
    print(f"\n📊 평가 결과:")
    print(f"   평가 제품 수: {eval_info['n_products_evaluated']:,}")
    print(f"   이상치율: {eval_info['anomaly_rate']*100:.2f}%")

print(f"\n⚡ 성능:")
print(f"   평균 탐지 시간: {avg_time_ms:.4f}ms")
print(f"   처리량: {throughput:,.0f} 건/초")

print("\n" + "=" * 60)
print("✅ Pickle 생성 및 검증 완료")
print("=" * 60)

Price Anomaly Model Pickle 생성 완료

📁 출력 파일: ..\pred\models\price_anomaly_v1.pkl
📊 파일 크기: 4.60 KB

📈 모델 정보:
   버전: 1.0.0
   생성일: 2025-12-15 01:44:15.166536
   카테고리 수: 13

⚙️ 하이퍼파라미터:
   warning_threshold: 2.5
   danger_threshold: 3.5
   use_log_transform: True
   mad_scaling_factor: 0.6745
   method: modified_z_score

📊 평가 결과:
   평가 제품 수: 1,600
   이상치율: 4.50%

⚡ 성능:
   평균 탐지 시간: 0.0010ms
   처리량: 995,090 건/초

✅ Pickle 생성 및 검증 완료


## 7. 사용 방법 예시

In [12]:
# 사용 방법 출력
usage_code = '''
# pred 서비스에서의 사용 방법

import pickle
import numpy as np

# 모델 로드
with open('pred/models/price_anomaly_v1.pkl', 'rb') as f:
    model = pickle.load(f)

def detect_anomaly(price, category_name=None):
    """가격 이상치 탐지"""
    components = model['components']
    hyperparams = model['hyperparameters']
    
    # 통계 선택
    if category_name in components['category_statistics']:
        stats = components['category_statistics'][category_name]
    else:
        stats = components['global_statistics']
    
    # Modified Z-Score 계산
    log_price = np.log10(max(1, price))
    median = stats['log_median']
    mad = stats['log_mad'] or 0.0001
    
    modified_zscore = 0.6745 * (log_price - median) / mad
    abs_zscore = abs(modified_zscore)
    
    # 레벨 판정
    if abs_zscore >= components['thresholds']['danger']:
        return {'level': 'danger', 'is_anomaly': True, 'zscore': modified_zscore}
    elif abs_zscore >= components['thresholds']['warning']:
        return {'level': 'warning', 'is_anomaly': True, 'zscore': modified_zscore}
    else:
        return {'level': 'normal', 'is_anomaly': False, 'zscore': modified_zscore}

# 사용 예시
result = detect_anomaly(50000, '음료')
print(f"이상치 여부: {result['is_anomaly']}, 레벨: {result['level']}")
'''

print("사용 방법:")
print(usage_code)

사용 방법:

# pred 서비스에서의 사용 방법

import pickle
import numpy as np

# 모델 로드
with open('pred/models/price_anomaly_v1.pkl', 'rb') as f:
    model = pickle.load(f)

def detect_anomaly(price, category_name=None):
    """가격 이상치 탐지"""
    components = model['components']
    hyperparams = model['hyperparameters']

    # 통계 선택
    if category_name in components['category_statistics']:
        stats = components['category_statistics'][category_name]
    else:
        stats = components['global_statistics']

    # Modified Z-Score 계산
    log_price = np.log10(max(1, price))
    median = stats['log_median']
    mad = stats['log_mad'] or 0.0001

    modified_zscore = 0.6745 * (log_price - median) / mad
    abs_zscore = abs(modified_zscore)

    # 레벨 판정
    if abs_zscore >= components['thresholds']['danger']:
        return {'level': 'danger', 'is_anomaly': True, 'zscore': modified_zscore}
    elif abs_zscore >= components['thresholds']['warning']:
        return {'level': 'warning', 'is_anomaly': True,

## 검증 체크리스트

### 필수 확인 항목
- [ ] 탐지기 상태 파일 로드 성공
- [ ] 데이터 일관성 검증 통과
- [ ] Pickle 파일 저장 완료
- [ ] 저장된 Pickle 로드 검증 통과
- [ ] 이상치 탐지 기능 테스트 성공
- [ ] 탐지 속도 < 1ms

### 다음 단계
1. **pred 서비스 코드 업데이트**: `pred/ml/price_anomaly.py`에서 이 Pickle 로드
2. **API 엔드포인트 테스트**: `/api/recommendations/price-check/` 호출 확인
3. **Phase 4 진행**: Adaptive Blending Orchestrator 구현